# Balanced SVM — Model restoration, evaluation, and diagnostic analysis

This notebook restores an existing English-only TF–IDF + class-weighted LinearSVC experiment and evaluates saved models without retraining them. It also compares previously exported balanced and unbalanced SVM predictions, audits multilingual label support, and records a follow-up experiment configuration.

**Execution environment:** Kaggle Python with the project input datasets mounted under `/kaggle/input/datasets/nizarmabrouk/`. Outputs are written to `/kaggle/working/nlp_setup/`.

**Required input datasets:**
- `nlp-semeval-2025-dati-e-setup`, including `kit_setup_nlp`, source archives, the frozen manifest, and reference audits.
- `nlp-svm-bilanciata`, including three fitted-model bundles, threshold search files, saved validation margins, and provenance metadata.
- For the paired system comparison, previously exported out-of-fold prediction files for **both** SVM variants must also be present in an attached dataset or ZIP file. The comparison step reports a missing-file error if either artifact cannot be located.

**How to run:** Start a fresh Kaggle session, attach the required datasets, and run cells in order. Saved artifacts are used instead of rerunning model fitting. The comparison section can be re-executed once both prediction artifacts are attached. The multilingual support audit and configuration persist their own outputs.

**Evidence and reproducibility:** The outputs displayed below were preserved from the source notebook. They document a prior Kaggle execution; the cleaned notebook has been syntax-checked but has not been re-executed against the mounted datasets. New execution should regenerate and save outputs before publication. The official test labels are not loaded.


## Input artifact discovery

Lists attached Kaggle datasets and locates the data preparation kit and saved SVM artifacts. This step does not load models or perform training.


In [3]:
from pathlib import Path
from zipfile import ZipFile, BadZipFile

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/nlp_setup")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not INPUT_ROOT.is_dir():
    raise FileNotFoundError(f"Kaggle input directory not found: {INPUT_ROOT}")

print("ATTACHED DATASETS:")
for folder in sorted(INPUT_ROOT.iterdir()):
    print(folder)

targets = {
    "nlp_setup.py",
    "split_manifest_v2.csv",
    "tfidf_models.joblib",
    "threshold_search.json",
    "validation_decision_scores.npz",
}

print("\nRELEVANT FILES:")
found = 0

for path in sorted(INPUT_ROOT.rglob("*")):
    if not path.is_file():
        continue

    if path.name in targets:
        print(path)
        found += 1

    elif path.suffix.lower() == ".zip":
        print("\nZIP:", path)
        try:
            with ZipFile(path) as archive:
                for member in archive.namelist():
                    if member.replace("\\", "/").split("/")[-1] in targets:
                        print("  →", member)
                        found += 1
        except BadZipFile:
            print("  ERROR: unreadable ZIP archive.")

print(f"\nArtifact scan complete: {found} relevant files found.")
print("No models loaded; no training performed.")

DATASET COLLEGATI:
/kaggle/input/datasets

FILE RILEVANTI:
/kaggle/input/datasets/nizarmabrouk/nlp-semeval-2025-dati-e-setup/kit_setup_nlp/nlp_setup.py
/kaggle/input/datasets/nizarmabrouk/nlp-semeval-2025-dati-e-setup/kit_setup_nlp/split_manifest_v2.csv
/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata/fold_1/tfidf_models.joblib
/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata/fold_1/threshold_search.json
/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata/fold_1/validation_decision_scores.npz
/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata/fold_2/tfidf_models.joblib
/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata/fold_2/threshold_search.json
/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata/fold_2/validation_decision_scores.npz
/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata/fold_3/tfidf_models.joblib
/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata/fold_3/threshold_search.json
/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata/fold_3/

## Restore data and validate experiment provenance

Loads the frozen taxonomy and split manifest, verifies dataset fingerprints and split integrity, and checks the balanced SVM configuration against saved training metadata.


In [4]:
from pathlib import Path
import importlib.util
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/nlp_setup')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = INPUT_ROOT / 'datasets/nizarmabrouk/nlp-semeval-2025-dati-e-setup'
KIT_DIR = DATA_ROOT / 'kit_setup_nlp'
SVMBAL_SOURCE_DIR = INPUT_ROOT / 'datasets/nizarmabrouk/nlp-svm-bilanciata'

def resume_read_json(path):
    return json.loads(path.read_text(encoding='utf-8-sig'))

def resume_load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

required = ['config.json', 'taxonomy.json', 'input_fingerprints.json',
            'split_manifest_v2.csv', 'status.json', 'validation_selection.csv']
required += [f'fold_{fold}/{filename}' for fold in (1, 2, 3) for filename in
             ('tfidf_models.joblib', 'threshold_search.json', 'validation_decision_scores.npz')]
missing = [name for name in required if not (SVMBAL_SOURCE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError('Incomplete training artifact dataset: ' + repr(missing))

setup = resume_load_module('nlp_setup_resume', KIT_DIR / 'nlp_setup.py')
taxonomy = resume_read_json(KIT_DIR / 'tassonomia_progetto_v1.json')
expected = resume_read_json(KIT_DIR / 'expected_data_fingerprints.json')
setup.require(taxonomy == resume_read_json(SVMBAL_SOURCE_DIR / 'taxonomy.json'),
              'Taxonomy does not match the saved training artifacts.')
saved_fingerprints = resume_read_json(SVMBAL_SOURCE_DIR / 'input_fingerprints.json')
for key in ('training_records', 'course_texts'):
    setup.require(saved_fingerprints[key] == expected[key], 'Saved training fingerprint mismatch: ' + key)

print('Loading data and validating fingerprints...', flush=True)
training_source = setup.locate_source(DATA_ROOT, 'training_NLP.zip', 'training')
test_source = setup.locate_source(DATA_ROOT, 'test_NLP.zip', 'test')
try:
    train_records = setup.load_training(training_source, taxonomy)
    course_texts = setup.load_course_texts(test_source)
finally:
    for source in (training_source, test_source):
        if source.archive is not None:
            source.archive.close()
setup.require(setup.fingerprint_records(train_records) == expected['training_records'], 'Training records fingerprint mismatch.')
setup.require(setup.fingerprint_records(course_texts) == expected['course_texts'], 'Test document fingerprint mismatch.')

# Reuse the frozen reference audit after verifying the input fingerprints.
# Preserve existing duplicate groups and exclusion decisions.
similarity = resume_read_json(KIT_DIR / 'near_duplicate_audit_reference.json')
setup.require('groups' in similarity and len(similarity['excluded_training_keys']) == 11,
              'Incomplete reference audit.')
manifest = setup.validate_manifest(train_records, similarity, KIT_DIR / 'split_manifest_v2.csv')
saved_manifest = pd.read_csv(SVMBAL_SOURCE_DIR / 'split_manifest_v2.csv')
pd.testing.assert_frame_equal(manifest.reset_index(drop=True), saved_manifest.reset_index(drop=True))

SVMBAL_CONFIG = resume_read_json(SVMBAL_SOURCE_DIR / 'config.json')
setup.require(SVMBAL_CONFIG['system'] == 'baseline_svmbal_EN_v1', 'Unexpected experiment identifier.')
setup.require(SVMBAL_CONFIG['seed'] == 42 and SVMBAL_CONFIG['multilingual'] is False, 'Unexpected training seed or language condition.')
setup.require(SVMBAL_CONFIG['split_version'] == 'v2', 'Unexpected split version.')
setup.require(SVMBAL_CONFIG['svm'] == {
    'C': 1.0, 'penalty': 'l2', 'loss': 'squared_hinge', 'dual': 'auto',
    'tol': 0.0001, 'max_iter': 10000, 'class_weight': 'balanced', 'random_state': 42},
    'Unexpected classifier parameters.')
setup.require(SVMBAL_CONFIG['vectorizer'] == {
    'max_features': 15000, 'analyzer': 'word', 'ngram_range': [1, 2], 'sublinear_tf': True},
    'Unexpected TF–IDF parameters.')
saved_status = resume_read_json(SVMBAL_SOURCE_DIR / 'status.json')
setup.require(saved_status['state'] == 'training_and_selection_complete', 'Saved model training is incomplete.')
setup.require(saved_status['outer_evaluated'] is False, 'Unexpected saved outer-evaluation status.')

rows = []
for fold in (1, 2, 3):
    parts = setup.make_fold(train_records, manifest, similarity, fold, False)
    row = {'fold': fold, **{part: len(records) for part, records in parts.items()}}
    with np.load(SVMBAL_SOURCE_DIR / f'fold_{fold}/validation_decision_scores.npz', allow_pickle=False) as scores:
        setup.require(scores['ids'].astype(str).tolist() == [r['document_id'] for r in parts['inner_validation']],
                      f'Validation document order mismatch in fold {fold}.')
        for level in ('coarse', 'fine'):
            setup.require(scores[level].shape == (len(parts['inner_validation']), len(taxonomy[level])),
                          'Saved decision score shape mismatch.')
            setup.require(not np.isnan(scores[level]).any(), 'NaN detected in saved decision scores.')
    rows.append(row)
setup.require([r['inner_train'] for r in rows] == [213, 212, 214], 'Unexpected inner-training counts.')
setup.require([r['outer_evaluation'] for r in rows] == [133, 134, 132], 'Unexpected outer-evaluation counts.')
official = resume_load_module('official_st2_resume', KIT_DIR / 'official_v3/subtask2_scorer.py')
official_classes = {level: (KIT_DIR / 'official_v3' / filename).read_text(encoding='utf-8').splitlines()
    for level, filename in [('coarse', 'subtask2_narratives.txt'), ('fine', 'subtask2_subnarratives.txt')]}

resume_summary = {
    'status': 'SVMBAL_RESUME_SETUP_OK', 'training_documents': len(train_records),
    'course_test_texts': len(course_texts), 'course_test_labels_loaded': False,
    'source_directory': str(SVMBAL_SOURCE_DIR), 'fold_sizes': rows,
    'similarity_audit': 'frozen kit reference reused after exact data fingerprint checks',
    'model_files_loaded': False, 'new_training_performed': False, 'outer_evaluation_performed': False,
    'environment': setup.environment_info(),
}
setup.save_json(OUTPUT_ROOT / 'svmbal_resume_setup_summary.json', resume_summary)
display(pd.DataFrame(rows))
print(json.dumps(resume_summary, indent=2, ensure_ascii=False))


Caricamento dati e verifica delle impronte...


,fold,inner_train,inner_validation,outer_evaluation
0,1,213,53,133
1,2,212,53,134
2,3,214,53,132


{
  "status": "SVMBAL_RESUME_SETUP_OK",
  "training_documents": 1699,
  "course_test_texts": 178,
  "course_test_labels_loaded": false,
  "source_directory": "/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata",
  "fold_sizes": [
    {
      "fold": 1,
      "inner_train": 213,
      "inner_validation": 53,
      "outer_evaluation": 133
    },
    {
      "fold": 2,
      "inner_train": 212,
      "inner_validation": 53,
      "outer_evaluation": 134
    },
    {
      "fold": 3,
      "inner_train": 214,
      "inner_validation": 53,
      "outer_evaluation": 132
    }
  ],
  "similarity_audit": "frozen kit reference reused after exact data fingerprint checks",
  "model_files_loaded": false,
  "new_training_performed": false,
  "outer_evaluation_performed": false,
  "environment": {
    "python": "3.13.18",
    "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
    "packages": {
      "numpy": "2.1.3",
      "pandas": "2.3.3",
      "scikit-learn": "1.6.1",
      "torch": "2.11.0

## Restore and verify fitted classifiers

Loads the previously fitted TF–IDF vectorizers and linear classifiers from the project-owned artifact dataset. Validation decision scores, threshold choices, class order, and model parameters are checked against stored artifacts. No refitting occurs.


In [5]:
import json
import hashlib
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.svm import LinearSVC
from IPython.display import display


def svmbal_restored_scores(models, X):
    columns = []
    for model in models:
        if isinstance(model, (float, np.floating)):
            assert model in (0., 1.)
            values = np.full(X.shape[0], np.inf if model == 1 else -np.inf)
        else:
            assert isinstance(model, LinearSVC)
            assert list(model.classes_) == [0, 1]
            values = model.decision_function(X)
            assert np.isfinite(values).all()
        columns.append(values)
    scores = np.column_stack(columns)
    assert not np.isnan(scores).any()
    assert (np.isfinite(scores).any(1) | np.isposinf(scores).any(1)).all()
    return scores


def svmbal_restored_decode(scores, threshold, classes):
    assert scores.shape[1] == len(classes)
    binary = (scores > threshold).astype(int)
    empty = np.flatnonzero(binary.sum(1) == 0)
    binary[empty, np.argmax(scores[empty], axis=1)] = 1
    return [[classes[j] for j in np.flatnonzero(row)] for row in binary]


def verify_restored_svmbal():
    required = ('setup', 'train_records', 'manifest', 'similarity', 'taxonomy',
                'SVMBAL_SOURCE_DIR', 'SVMBAL_CONFIG', 'OUTPUT_ROOT', 'resume_summary')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError('Restore the experiment setup before loading models. Missing: ' + repr(missing))
    assert resume_summary['status'] == 'SVMBAL_RESUME_SETUP_OK'
    selections = pd.read_csv(SVMBAL_SOURCE_DIR / 'validation_selection.csv')
    assert len(selections) == 6
    bundles, rows, fingerprints = {}, [], {}
    for fold in (1, 2, 3):
        print(f'Fold {fold}/3: restoring models and verifying validation outputs...', flush=True)
        folder = SVMBAL_SOURCE_DIR / f'fold_{fold}'
        path = folder / 'tfidf_models.joblib'
        fingerprints[str(fold)] = hashlib.sha256(path.read_bytes()).hexdigest()
        # Deserialize only project-owned model bundles from the specified artifact dataset.
        with warnings.catch_warnings():
            warnings.simplefilter('error', InconsistentVersionWarning)
            bundle = joblib.load(path)
        parts = setup.make_fold(train_records, manifest, similarity, fold, False)
        assert bundle['train_ids'] == [r['document_id'] for r in parts['inner_train']]
        assert bundle['validation_ids'] == [r['document_id'] for r in parts['inner_validation']]
        held = {r['document_id'] for r in parts['outer_evaluation']}
        assert not held.intersection(bundle['train_ids'] + bundle['validation_ids'])
        vectorizer_params = bundle['vectorizer'].get_params()
        for key, value in SVMBAL_CONFIG['vectorizer'].items():
            assert vectorizer_params[key] == (tuple(value) if key == 'ngram_range' else value)
        assert vectorizer_params['norm'] == 'l2'
        X = bundle['vectorizer'].transform([r['text'] for r in parts['inner_validation']])
        search = json.loads((folder / 'threshold_search.json').read_text(encoding='utf-8'))
        with np.load(folder / 'validation_decision_scores.npz', allow_pickle=False) as stored:
            assert stored['ids'].astype(str).tolist() == bundle['validation_ids']
            for level in ('coarse', 'fine'):
                assert bundle['classes'][level] == taxonomy[level]
                models = bundle['models'][level]
                assert len(models) == len(taxonomy[level])
                y = setup.binary_labels([r[level] for r in parts['inner_train']], taxonomy[level])
                for j, model in enumerate(models):
                    target_values = np.unique(y[:, j])
                    if len(target_values) == 1:
                        assert isinstance(model, (float, np.floating)) and model == float(target_values[0])
                    else:
                        assert isinstance(model, LinearSVC)
                        assert all(model.get_params()[k] == v for k, v in SVMBAL_CONFIG['svm'].items())
                current = svmbal_restored_scores(models, X)
                old = stored[level]
                assert np.array_equal(np.isposinf(current), np.isposinf(old))
                assert np.array_equal(np.isneginf(current), np.isneginf(old))
                assert np.allclose(current, old, rtol=1e-7, atol=1e-9), 'Validation decision scores could not be reproduced.'
                best = max(search[level], key=lambda r: (r['validation_sample_f1'], r['threshold']))
                threshold = bundle['thresholds'][level]
                assert threshold == best['threshold']
                assert [r['threshold'] for r in search[level]] == SVMBAL_CONFIG['threshold_grid']
                predicted = svmbal_restored_decode(current, threshold, taxonomy[level])
                score = setup.sample_f1_strict([r[level] for r in parts['inner_validation']], predicted, taxonomy[level])['mean']
                saved_row = selections[(selections.fold == fold) & (selections.level == level)]
                assert len(saved_row) == 1
                assert np.isclose(float(saved_row.iloc[0]['threshold']), threshold, rtol=0, atol=1e-12)
                assert np.isclose(score, best['validation_sample_f1'], rtol=0, atol=1e-12)
                assert np.isclose(score, float(saved_row.iloc[0]['validation_F1']), rtol=0, atol=1e-12)
                rows.append({'fold': fold, 'level': level, 'threshold': threshold,
                             'validation_F1_reproduced': score, 'saved_scores_match': True})
        bundles[fold] = bundle
    summary = {'status': 'SVMBAL_MODELS_VERIFIED_OK', 'folds': 3, 'seed': 42,
               'validation_scores_reproduced': True, 'thresholds_changed': False,
               'new_training_performed': False, 'outer_evaluation_performed': False,
               'model_sha256': fingerprints, 'source_directory': str(SVMBAL_SOURCE_DIR)}
    setup.save_json(OUTPUT_ROOT / 'svmbal_restored_models_summary.json', summary)
    pd.DataFrame(rows).to_csv(OUTPUT_ROOT / 'svmbal_restored_validation.csv', index=False)
    display(pd.DataFrame(rows))
    print(json.dumps(summary, indent=2))
    return bundles, summary


svmbal_verified_bundles, svmbal_verification_result = verify_restored_svmbal()


Fold 1/3: caricamento e verifica sulla validazione...
Fold 2/3: caricamento e verifica sulla validazione...
Fold 3/3: caricamento e verifica sulla validazione...


,fold,level,threshold,validation_F1_reproduced,saved_scores_match
0,1,coarse,-0.4,0.544744,True
1,1,fine,-0.6,0.482664,True
2,2,coarse,-0.2,0.571698,True
3,2,fine,-0.5,0.473389,True
4,3,coarse,-0.4,0.503863,True
5,3,fine,-0.5,0.447126,True


{
  "status": "SVMBAL_MODELS_VERIFIED_OK",
  "folds": 3,
  "seed": 42,
  "validation_scores_reproduced": true,
  "thresholds_changed": false,
  "new_training_performed": false,
  "outer_evaluation_performed": false,
  "model_sha256": {
    "1": "33c7a7c264ae9ac5b9b2168d0cb95b73e91391e5a20654493a7f6b8310912b22",
    "2": "a681dec0386ea6c13196cb4b3030d3e68ecdb750acaed6ba1f58f71ad0d3bf9b",
    "3": "88b3aa93d155979b645c292306230eb3a0c1502cad25c023c2072be927390381"
  },
  "source_directory": "/kaggle/input/datasets/nizarmabrouk/nlp-svm-bilanciata"
}


## Evaluate the frozen models on outer folds

Applies verified, fixed-threshold classifiers to held-out English documents. Reports sample-F1 and additional diagnostics; exports per-fold decisions, predictions, metrics, environment metadata, and a ZIP archive.


In [6]:
from pathlib import Path
import json
import time
import uuid
import hashlib
import numpy as np
import pandas as pd
from zipfile import ZipFile, ZIP_DEFLATED
from sklearn.metrics import f1_score, precision_score, recall_score
from IPython.display import display

required = ('setup', 'train_records', 'manifest', 'similarity', 'taxonomy', 'official',
    'official_classes', 'SVMBAL_CONFIG', 'SVMBAL_SOURCE_DIR', 'OUTPUT_ROOT',
    'svmbal_verified_bundles', 'svmbal_verification_result', 'svmbal_restored_scores', 'svmbal_restored_decode')
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError('Run the preceding setup and verification cells. Missing: ' + repr(missing))
assert svmbal_verification_result['status'] == 'SVMBAL_MODELS_VERIFIED_OK'
assert set(svmbal_verified_bundles) == {1, 2, 3}
for fold in (1, 2, 3):
    path = SVMBAL_SOURCE_DIR / f'fold_{fold}/tfidf_models.joblib'
    assert hashlib.sha256(path.read_bytes()).hexdigest() == svmbal_verification_result['model_sha256'][str(fold)]
    saved = json.loads((SVMBAL_SOURCE_DIR / f'fold_{fold}/threshold_search.json').read_text(encoding='utf-8'))
    for level in ('coarse','fine'):
        best = max(saved[level], key=lambda r: (r['validation_sample_f1'], r['threshold']))
        assert svmbal_verified_bundles[fold]['thresholds'][level] == best['threshold']
SVMBAL_EVAL_DIR = OUTPUT_ROOT / 'evaluations' / ('svmbal_EN_v1_seed42_' + uuid.uuid4().hex[:10])
SVMBAL_EVAL_DIR.mkdir(parents=True, exist_ok=False)
setup.save_json(SVMBAL_EVAL_DIR/'config.json', SVMBAL_CONFIG)
setup.save_json(SVMBAL_EVAL_DIR/'taxonomy.json', taxonomy)
setup.save_json(SVMBAL_EVAL_DIR/'model_verification.json', svmbal_verification_result)
manifest.to_csv(SVMBAL_EVAL_DIR/'split_manifest_v2.csv',index=False)
print('Three model bundles verified; selected thresholds are frozen. Running CPU outer-fold evaluation.', flush=True)

def svmbal_write_predictions(path, records, coarse, fine):
    setup.require(len(records) == len(coarse) == len(fine), 'Incomplete prediction arrays.')
    with open(path, 'w', encoding='utf-8', newline='') as f:
        for row, c, s in zip(records, coarse, fine):
            f.write(row['document_id']+'\t'+';'.join(c)+'\t'+';'.join(s)+'\n')

def svmbal_evaluation_metrics(records, predicted, official_classes):
    result = {}
    for level in ('coarse', 'fine'):
        gold = [r[level] for r in records]
        pred = predicted[level]
        classes = taxonomy[level]
        y = setup.binary_labels(gold, classes)
        p = setup.binary_labels(pred, classes)
        result['F1_'+level] = setup.sample_f1_strict(gold, pred, classes)['mean']
        result['mean_predicted_'+level] = float(p.sum(axis=1).mean())
        allowed = set(official_classes[level])
        gold_v3 = [[l for l in ls if l in allowed] for ls in gold]
        pred_v3 = [[l for l in ls if l in allowed] for ls in pred]
        result['F1_'+level+'_v3'] = float(official.evaluate(gold, pred, official_classes[level])[0])
        # Check explicit projection agrees with the original scorer's silent ignore behavior.
        assert np.isclose(result['F1_'+level+'_v3'], setup.sample_f1_strict(gold_v3, pred_v3, official_classes[level])['mean'])
        result['gold_labels_ignored_v3_'+level] = sum(len(set(ls)-allowed) for ls in gold)
        result['pred_labels_ignored_v3_'+level] = sum(len(set(ls)-allowed) for ls in pred)
        if level == 'fine':
            supported = y.sum(axis=0) > 0
            result['micro_F1_fine'] = float(f1_score(y, p, average='micro', zero_division=0))
            result['macro_F1_fine_gold_supported'] = float(f1_score(y[:,supported],p[:,supported],average='macro',zero_division=0))
            result['macro_fine_class_count'] = int(supported.sum())
            result['micro_precision_fine'] = float(precision_score(y,p,average='micro',zero_division=0))
            result['micro_recall_fine'] = float(recall_score(y,p,average='micro',zero_division=0))
    fine_specific = sum(l != 'Other' for ls in predicted['fine'] for l in ls)
    orphan = sum(l != 'Other' and l.rsplit(': ',1)[0] not in coarse
                 for coarse, fine in zip(predicted['coarse'],predicted['fine']) for l in fine)
    result['fine_without_parent_rate'] = orphan / fine_specific if fine_specific else 0.0
    for level in ('coarse','fine'):
        result['mixed_Other_rate_'+level] = float(np.mean(['Other' in ls and len(ls)>1 for ls in predicted[level]]))
    return result

systems = ['Other_constant','TFIDF_word_SVM_balanced_threshold_0','TFIDF_word_SVM_balanced_threshold_tuned']
all_predictions = {system:[] for system in systems}
all_outer = []
for fold in (1,2,3):
    data = setup.make_fold(train_records,manifest,similarity,fold,multilingual=False)
    outer = data['outer_evaluation']; fold_dir=SVMBAL_EVAL_DIR / f'fold_{fold}'
    fold_dir.mkdir()
    # Load only models created by this run; never load untrusted joblib files.
    bundle = svmbal_verified_bundles[fold]
    setup.require(all(bundle['classes'][k]==taxonomy[k] for k in ('coarse','fine')), 'Class order mismatch.')
    held_ids={r['document_id'] for r in outer}
    setup.require(not held_ids.intersection(bundle['train_ids']+bundle['validation_ids']), 'Train/held-out document overlap detected.')
    started=time.perf_counter()
    X_outer=bundle['vectorizer'].transform([r['text'] for r in outer])
    outer_probs={k:svmbal_restored_scores(bundle['models'][k],X_outer) for k in ('coarse','fine')}
    elapsed=time.perf_counter()-started
    setup.save_json(fold_dir/'outer_inference_timing.json',{'vectorize_and_predict_seconds':elapsed,'documents':len(outer)})
    np.savez_compressed(fold_dir/'outer_decision_scores.npz',ids=np.array([r['document_id'] for r in outer]),**outer_probs)
    all_outer.extend(outer)
    for system in systems:
        predicted={}
        for level in ('coarse','fine'):
            if system=='Other_constant':predicted[level]=[['Other'] for _ in outer]
            else:
                threshold=0. if system=='TFIDF_word_SVM_balanced_threshold_0' else bundle['thresholds'][level]
                predicted[level]=svmbal_restored_decode(outer_probs[level],threshold,taxonomy[level])
        svmbal_write_predictions(fold_dir/(system+'.tsv'),outer,predicted['coarse'],predicted['fine'])
        all_predictions[system].extend([{'id':r['document_id'],'fold':fold,'coarse':c,'fine':f}
            for r,c,f in zip(outer,predicted['coarse'],predicted['fine'])])
setup.require(len(all_outer)==399 and len({r['document_id'] for r in all_outer})==399,'Incomplete outer-fold coverage.')
all_outer=sorted(all_outer,key=lambda r:r['document_id'])
metric_rows=[]
per_document=[]
for system in systems:
    ordered=sorted(all_predictions[system],key=lambda r:r['id'])
    setup.require([r['id'] for r in ordered]==[r['document_id'] for r in all_outer], 'Prediction/document ID mismatch.')
    setup.save_json(SVMBAL_EVAL_DIR/(system+'_predictions.json'), ordered)
    predicted={level:[r[level] for r in ordered] for level in ('coarse','fine')}
    metric_rows.append({'system':system,'n_documents':399,**svmbal_evaluation_metrics(all_outer,predicted,official_classes)})
    svmbal_write_predictions(SVMBAL_EVAL_DIR/(system+'_out_of_fold.tsv'),all_outer,predicted['coarse'],predicted['fine'])
    for gold,pred in zip(all_outer,ordered):
        row={'system':system,'id':gold['document_id'],'fold':pred['fold']}
        for level in ('coarse','fine'):
            row['F1_'+level]=setup.sample_f1_strict([gold[level]],[pred[level]],taxonomy[level])['mean']
        per_document.append(row)
metrics_table=pd.DataFrame(metric_rows)
metrics_table.to_csv(SVMBAL_EVAL_DIR/'metrics.csv',index=False)
pd.DataFrame(per_document).to_csv(SVMBAL_EVAL_DIR/'per_document_scores.csv',index=False)
setup.save_json(SVMBAL_EVAL_DIR/'metrics.json',metric_rows)
setup.save_json(SVMBAL_EVAL_DIR/'status.json',{'state':'complete','outer_evaluated':True,'course_test_evaluated':False})
display(metrics_table[['system','n_documents','F1_coarse','F1_fine','F1_fine_v3','micro_F1_fine','macro_F1_fine_gold_supported']])
print('EVALUATION_OK — 399 outer-fold predictions per system; no threshold tuning on held-out results.')

assert all(np.isclose(metric_rows[0]['F1_'+level], 0.42355889724310775, atol=1e-12, rtol=0) for level in ('coarse','fine'))
svmbal_evaluation_result = {
    'status': 'SVMBAL_EVALUATION_OK', 'outer_documents': 399, 'seed': 42,
    'systems': metric_rows, 'output_directory': str(SVMBAL_EVAL_DIR),
    'aggregation': 'concatenated outer predictions, each document once per system',
    'thresholds_changed': False, 'new_training_performed': False,
    'course_test_labels_loaded': False, 'external_tuning_performed': False,
    'limitations': ['Exploratory experiment motivated by prior outer error analysis.',
                    'One training seed; no claim of statistically significant improvement.']
}
setup.save_json(SVMBAL_EVAL_DIR/'evaluation_summary.json',svmbal_evaluation_result)
setup.save_json(SVMBAL_EVAL_DIR/'environment.json',setup.environment_info())
archive_path = SVMBAL_EVAL_DIR.with_suffix('.zip')
with ZipFile(archive_path,'w',ZIP_DEFLATED) as archive:
    for path in sorted(SVMBAL_EVAL_DIR.rglob('*')):
        if path.is_file():
            archive.write(path,path.relative_to(SVMBAL_EVAL_DIR).as_posix())
with ZipFile(archive_path) as archive:
    assert archive.testzip() is None
print(json.dumps(svmbal_evaluation_result,indent=2,ensure_ascii=False))
print('Evaluation archive:',archive_path)


Tre modelli verificati; soglie congelate. Valutazione esterna CPU.


,system,n_documents,F1_coarse,F1_fine,F1_fine_v3,micro_F1_fine,macro_F1_fine_gold_supported
0,Other_constant,399,0.423559,0.423559,0.423559,0.265306,0.007084
1,TFIDF_word_SVM_balanced_threshold_0,399,0.490548,0.436526,0.436685,0.300389,0.032463
2,TFIDF_word_SVM_balanced_threshold_tuned,399,0.549758,0.461673,0.462045,0.364345,0.118913


VALUTAZIONE_OK — 399 predizioni esterne per sistema; nessun tuning sui punteggi esterni.
{
  "status": "SVMBAL_EVALUATION_OK",
  "outer_documents": 399,
  "seed": 42,
  "systems": [
    {
      "system": "Other_constant",
      "n_documents": 399,
      "F1_coarse": 0.42355889724310775,
      "mean_predicted_coarse": 1.0,
      "F1_coarse_v3": 0.42355889724310775,
      "gold_labels_ignored_v3_coarse": 0,
      "pred_labels_ignored_v3_coarse": 0,
      "F1_fine": 0.42355889724310775,
      "mean_predicted_fine": 1.0,
      "F1_fine_v3": 0.42355889724310775,
      "gold_labels_ignored_v3_fine": 5,
      "pred_labels_ignored_v3_fine": 0,
      "micro_F1_fine": 0.2653061224489796,
      "macro_F1_fine_gold_supported": 0.007084171696847753,
      "macro_fine_class_count": 84,
      "micro_precision_fine": 0.42355889724310775,
      "micro_recall_fine": 0.19314285714285714,
      "fine_without_parent_rate": 0.0,
      "mixed_Other_rate_coarse": 0.0,
      "mixed_Other_rate_fine": 0.0
    },

## Compare balanced and unbalanced SVM systems

Reconstructs out-of-fold predictions for both systems and compares their document-level F1 using a paired bootstrap over leakage groups. The intervals are exploratory and conditional on one training seed; intervals crossing zero do not establish a reliable performance gain.


In [7]:
from pathlib import Path
from zipfile import ZipFile, BadZipFile, ZIP_DEFLATED
import hashlib
import json
import uuid
import numpy as np
import pandas as pd
from IPython.display import display

def paired_group_bootstrap(scores, groups, n_resamples=2000, seed=20261104):
    # scores: documents x models x levels. Same sampled indices for all columns.
    scores = np.asarray(scores, dtype=np.float64)
    assert scores.ndim == 3 and len(scores) == len(groups) and np.isfinite(scores).all()
    unique, inverse = np.unique(groups, return_inverse=True)
    sizes = np.bincount(inverse)
    sums = np.zeros((len(unique),) + scores.shape[1:], dtype=np.float64)
    np.add.at(sums, inverse, scores)
    rng = np.random.default_rng(seed)
    samples = np.empty((n_resamples,) + scores.shape[1:], dtype=np.float64)
    document_counts = np.empty(n_resamples, dtype=np.int64)
    for b in range(n_resamples):
        chosen = rng.integers(0, len(unique), size=len(unique))
        # Large groups retain their document weight. Denominator can vary.
        document_counts[b] = sizes[chosen].sum()
        samples[b] = sums[chosen].sum(axis=0) / document_counts[b]
    return samples, document_counts

def balance_find_prediction(filename):
    found = {}
    for root in (Path('/kaggle/input'), Path('/kaggle/working/nlp_setup')):
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if not path.is_file():
                continue
            candidates = []
            if path.name == filename:
                candidates.append((str(path), path.read_bytes()))
            elif path.suffix.lower() == '.zip':
                try:
                    with ZipFile(path) as archive:
                        for member in archive.namelist():
                            if member.replace('\\', '/').split('/')[-1] == filename:
                                candidates.append((str(path) + ' :: ' + member, archive.read(member)))
                except BadZipFile:
                    continue
            for origin, raw in candidates:
                found.setdefault(hashlib.sha256(raw).hexdigest(), (origin, raw))
    if not found:
        raise FileNotFoundError('Missing ' + filename + '. Attach the matching saved prediction artifacts; retraining is not required.')
    if len(found) != 1:
        raise RuntimeError('Copie differenti dello stesso file: ' + repr([v[0] for v in found.values()]))
    digest, (origin, raw) = next(iter(found.items()))
    return raw, {'source': origin, 'sha256': digest}

def balance_band(n):
    return '0' if n == 0 else '1' if n == 1 else '2–4' if n <= 4 else '5–9' if n <= 9 else '10+'

def compare_balanced_svm():
    required = ('setup', 'train_records', 'manifest', 'similarity', 'taxonomy')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError('Restore the experiment setup first. Missing: ' + repr(missing))
    filenames = {'SVM_originale': 'TFIDF_word_SVM_threshold_tuned_out_of_fold.tsv',
                 'SVM_bilanciata': 'TFIDF_word_SVM_balanced_threshold_tuned_predictions.json'}
    expected = {'SVM_originale': [0.5402544962695339, 0.45857780727307484],
                'SVM_bilanciata': [0.5497580531415118, 0.46167303874070786]}
    records = sorted([r for r in train_records if r['language'] == 'EN'], key=lambda r: r['document_id'])
    ids = [r['document_id'] for r in records]
    assert len(ids) == len(set(ids)) == 399
    external = manifest[(manifest.language == 'EN') & (manifest.partition == 'outer_evaluation')].set_index('document_id')
    assert len(external) == 399 and external.index.is_unique
    folds = external.loc[ids, 'fold'].to_numpy(dtype=int)
    groups = external.loc[ids, 'leakage_group_id'].astype(str).to_numpy()
    assert len(set(groups)) == 395
    assert all(len(set(folds[groups == g])) == 1 for g in set(groups))
    names = list(filenames)
    gold = {level: setup.binary_labels([r[level] for r in records], taxonomy[level]) for level in ('coarse', 'fine')}
    predictions, scores, provenance, metric_rows = {}, [], {}, []
    for name, filename in filenames.items():
        raw, provenance[name] = balance_find_prediction(filename)
        if filename.endswith('.json'):
            rows = json.loads(raw.decode('utf-8-sig'))
        else:
            rows = []
            for line in raw.decode('utf-8-sig').splitlines():
                if not line.strip():
                    continue
                fields = line.split('\t')
                assert len(fields) == 3
                rows.append({'id': fields[0], 'coarse': fields[1].split(';') if fields[1] else [],
                             'fine': fields[2].split(';') if fields[2] else []})
        rows = sorted(rows, key=lambda r: r['id'])
        assert [r['id'] for r in rows] == ids
        if filename.endswith('.json'):
            assert [int(r['fold']) for r in rows] == folds.tolist()
        predictions[name], level_scores = {}, []
        for j, level in enumerate(('coarse', 'fine')):
            assert all(len(r[level]) == len(set(r[level])) for r in rows)
            p = setup.binary_labels([r[level] for r in rows], taxonomy[level])
            predictions[name][level] = p
            y = gold[level]
            denom = y.sum(1) + p.sum(1)
            f1 = np.divide(2 * (y * p).sum(1), denom, out=np.zeros(399), where=denom != 0)
            assert np.isclose(f1.mean(), expected[name][j], atol=1e-9, rtol=0), 'Previously recorded F1 could not be reproduced.'
            level_scores.append(f1)
            metric_rows.append({'system': name, 'level': level, 'F1': float(f1.mean())})
        scores.append(np.stack(level_scores, axis=1))
    scores = np.stack(scores, axis=1)  # Documents × systems × label levels.
    print('Saved scores reproduced. Paired bootstrap: 2,000 resamples across 395 groups.', flush=True)
    samples, counts = paired_group_bootstrap(scores, groups, 2000, 20261104)
    differences = []
    for j, level in enumerate(('coarse', 'fine')):
        delta = samples[:, 1, j] - samples[:, 0, j]
        lo, hi = np.quantile(delta, [.025, .975])
        differences.append({'level': level, 'delta_bilanciata_meno_originale': float((scores[:, 1, j] - scores[:, 0, j]).mean()),
                            'CI95_low': float(lo), 'CI95_high': float(hi), 'includes_zero': bool(lo <= 0 <= hi)})
    support = np.zeros_like(gold['fine'])
    for fold in (1, 2, 3):
        train = setup.make_fold(train_records, manifest, similarity, fold, False)['inner_train']
        support[folds == fold] = setup.binary_labels([r['fine'] for r in train], taxonomy['fine']).sum(0)
    bands = np.array([[balance_band(int(n)) for n in row] for row in support])
    specific = np.ones_like(gold['fine'], dtype=bool)
    specific[:, taxonomy['fine'].index('Other')] = False
    support_rows = []
    for name in names:
        y, p = gold['fine'], predictions[name]['fine']
        for band in ('0', '1', '2–4', '5–9', '10+'):
            mask = specific & (bands == band)
            tp = int(((y == 1) & (p == 1) & mask).sum())
            fn = int(((y == 1) & (p == 0) & mask).sum())
            fp = int(((y == 0) & (p == 1) & mask).sum())
            support_rows.append({'system': name, 'support_band': band, 'gold_occurrences': tp + fn,
                'TP': tp, 'FN': fn, 'FP': fp, 'recall': tp/(tp+fn) if tp+fn else None,
                'precision': tp/(tp+fp) if tp+fp else None})
    dest = Path('/kaggle/working/nlp_setup/comparisons') / ('svm_balancing_seed42_' + uuid.uuid4().hex[:10])
    dest.mkdir(parents=True, exist_ok=False)
    for filename, rows in [('metrics', metric_rows), ('paired_differences', differences), ('training_support', support_rows)]:
        pd.DataFrame(rows).to_csv(dest / (filename + '.csv'), index=False)
        print('\n' + filename)
        display(pd.DataFrame(rows))
    np.savez_compressed(dest / 'bootstrap.npz', samples=samples, sampled_documents=counts,
                        systems=np.array(names), levels=np.array(['coarse', 'fine']))
    payload = dict(ids=np.array(ids), folds=folds, groups=groups, systems=np.array(names), document_scores=scores,
                   gold_coarse=gold['coarse'], gold_fine=gold['fine'], inner_train_fine_support=support)
    for level in ('coarse', 'fine'):
        payload['predicted_' + level] = np.stack([predictions[n][level] for n in names])
        payload['classes_' + level] = np.array(taxonomy[level])
    np.savez_compressed(dest / 'comparison_predictions.npz', **payload)
    summary = {'status': 'SVMBAL_COMPARISON_OK', 'documents': 399, 'groups': 395,
        'resamples': 2000, 'bootstrap_seed': 20261104, 'training_seeds': [42],
        'difference_direction': 'balanced minus original', 'paired_differences': differences,
        'bootstrap_method': 'paired group percentile, document-weighted sample F1; non-stratified by fold',
        'source': provenance, 'output_directory': str(dest), 'new_training_performed': False,
        'thresholds_changed': False,
        'limitations': ['Exploratory post-selection comparison; no correction for model selection.',
            'Conditional on fitted models and fixed splits, one training seed only.',
            'Group resampling does not capture all dependence from overlapping CV training sets.',
            'Marginal 95% intervals for sample F1 only, not equivalence tests.',
            'Support-band precision and recall are descriptive, without confidence intervals.',
            'Support counts inner-training documents; aggregate counts are document-label occurrences.',
            'Only global Other is excluded from support analysis; 10+ does not mean sufficient data.']}
    (dest / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    with ZipFile(dest.with_suffix('.zip'), 'w', ZIP_DEFLATED) as archive:
        for path in sorted(dest.iterdir()):
            archive.write(path, path.name)
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print('Saved artifact archive:', dest.with_suffix('.zip'))
    return summary

svmbal_comparison_result = compare_balanced_svm()


Punteggi riprodotti. Bootstrap appaiato: 2000 ricampionamenti di 395 gruppi.

metrics


,system,level,F1
0,SVM_originale,coarse,0.540254
1,SVM_originale,fine,0.458578
2,SVM_bilanciata,coarse,0.549758
3,SVM_bilanciata,fine,0.461673



paired_differences


,level,delta_bilanciata_meno_originale,CI95_low,CI95_high,includes_zero
0,coarse,0.009504,-0.007815,0.027763,True
1,fine,0.003095,-0.014056,0.018813,True



training_support


,system,support_band,gold_occurrences,TP,FN,FP,recall,precision
0,SVM_originale,0,19,0,19,0,0.000000,NaN
1,SVM_originale,1,45,0,45,2,0.000000,0.000000
2,SVM_originale,2–4,165,1,164,7,0.006061,0.125000
3,SVM_originale,5–9,269,30,239,73,0.111524,0.291262
4,SVM_originale,10+,208,86,122,176,0.413462,0.328244
5,SVM_bilanciata,0,19,0,19,0,0.000000,NaN
6,SVM_bilanciata,1,45,0,45,2,0.000000,0.000000
7,SVM_bilanciata,2–4,165,4,161,9,0.024242,0.307692
8,SVM_bilanciata,5–9,269,52,217,117,0.193309,0.307692
9,SVM_bilanciata,10+,208,104,104,262,0.500000,0.284153


{
  "status": "SVMBAL_COMPARISON_OK",
  "documents": 399,
  "groups": 395,
  "resamples": 2000,
  "bootstrap_seed": 20261104,
  "training_seeds": [
    42
  ],
  "difference_direction": "balanced minus original",
  "paired_differences": [
    {
      "level": "coarse",
      "delta_bilanciata_meno_originale": 0.009503556871977925,
      "CI95_low": -0.007814934563893105,
      "CI95_high": 0.027762972371559887,
      "includes_zero": true
    },
    {
      "level": "fine",
      "delta_bilanciata_meno_originale": 0.0030952314676330594,
      "CI95_low": -0.014055887319905255,
      "CI95_high": 0.01881257121131527,
      "includes_zero": true
    }
  ],
  "bootstrap_method": "paired group percentile, document-weighted sample F1; non-stratified by fold",
  "source": {
    "SVM_originale": {
      "source": "/kaggle/input/datasets/nizarmabrouk/nlp-svm-originale-valutata/TFIDF_word_SVM_threshold_tuned_out_of_fold.tsv",
      "sha256": "bf35a0661ccb9c5d91167cb86af74f7e5e1ebcadd147fcecc879

## Audit multilingual training support

Counts label support by language and fold without changing held-out partitions or training any model. This descriptive analysis identifies rare categories and remaining coverage gaps.


In [8]:
from pathlib import Path
from collections import Counter
from zipfile import ZipFile, ZIP_DEFLATED
import json
import uuid
import numpy as np
import pandas as pd
from IPython.display import display


def multi_support_counts(records, classes):
    index = {label: i for i, label in enumerate(classes)}
    result = np.zeros(len(classes), dtype=int)
    for record in records:
        labels = set(record['fine'])
        assert labels <= set(index)
        for label in labels:
            result[index[label]] += 1
    return result


def audit_multilingual_support():
    required = ('setup', 'train_records', 'manifest', 'similarity', 'taxonomy')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError('Restore the experiment setup before the support audit. Missing: ' + repr(missing))
    classes = taxonomy['fine']
    assert len(classes) == 96 and len(set(classes)) == 96
    specific = np.array([label != 'Other' for label in classes])
    detail, overview, language_rows = [], [], []
    for fold in (1, 2, 3):
        english = setup.make_fold(train_records, manifest, similarity, fold, False)
        multilingual = setup.make_fold(train_records, manifest, similarity, fold, True)
        for partition in ('inner_validation', 'outer_evaluation'):
            assert [r['key'] for r in english[partition]] == [r['key'] for r in multilingual[partition]]
            assert all(r['language'] == 'EN' for r in multilingual[partition])
        en = english['inner_train']
        multi = multilingual['inner_train']
        en_keys = {r['key'] for r in en}
        multi_keys = {r['key'] for r in multi}
        assert len(en_keys) == len(en) and len(multi_keys) == len(multi)
        assert en_keys <= multi_keys
        assert {r['key'] for r in multi if r['language'] == 'EN'} == en_keys
        held = {r['key'] for p in ('inner_validation', 'outer_evaluation') for r in english[p]}
        assert not held & multi_keys
        assert not set(similarity['excluded_training_keys']) & multi_keys
        for members in similarity['groups'].values():
            assert not (held.intersection(members) and multi_keys.intersection(members)), 'Duplicate group spans training and held-out partitions.'
        a = multi_support_counts(en, classes)
        b = multi_support_counts(multi, classes)
        assert np.all(b >= a)
        by_language = {}
        for language in ('EN', 'BG', 'HI', 'PT', 'RU'):
            subset = [r for r in multi if r['language'] == language]
            by_language[language] = multi_support_counts(subset, classes)
            language_rows.append({'fold': fold, 'language': language, 'training_documents': len(subset)})
        assert np.array_equal(sum(by_language.values()), b)
        rare = specific & (a >= 1) & (a <= 4)
        overview.append({'fold': fold, 'EN_train': len(en), 'multilingual_train': len(multi),
            'auxiliary_documents': len(multi) - len(en),
            'specific_labels_seen_EN': int(((a > 0) & specific).sum()),
            'specific_labels_seen_multilingual': int(((b > 0) & specific).sum()),
            'unseen_EN_become_seen': int(((a == 0) & (b > 0) & specific).sum()),
            'still_unseen_multilingual': int(((b == 0) & specific).sum()),
            'labels_with_1_to_4_EN_examples': int(rare.sum()),
            'of_these_reach_5plus': int((rare & (b >= 5)).sum()),
            'of_these_reach_10plus': int((rare & (b >= 10)).sum())})
        for j, label in enumerate(classes):
            detail.append({'fold': fold, 'label': label, 'EN_support': int(a[j]),
                'multilingual_support': int(b[j]), 'added_support': int(b[j] - a[j]),
                **{lang + '_support': int(by_language[lang][j]) for lang in ('BG', 'HI', 'PT', 'RU')}})
    assert [r['EN_train'] for r in overview] == [213, 212, 214]
    assert [r['multilingual_train'] for r in overview] == [1502, 1501, 1503]
    per_label = pd.DataFrame(detail)
    specific_rows = per_label[per_label.label != 'Other']
    across = specific_rows.groupby('label', sort=True).agg(
        EN_support_min=('EN_support', 'min'), EN_support_max=('EN_support', 'max'),
        multilingual_support_min=('multilingual_support', 'min'),
        multilingual_support_max=('multilingual_support', 'max'),
        added_support_min=('added_support', 'min')).reset_index()
    priority = across[across.EN_support_min <= 4].sort_values(
        ['added_support_min', 'label'], ascending=[False, True])
    unseen = per_label[(per_label.multilingual_support == 0) & (per_label.label != 'Other')]
    dest = Path('/kaggle/working/nlp_setup/multilingual') / ('support_audit_' + uuid.uuid4().hex[:10])
    dest.mkdir(parents=True, exist_ok=False)
    for filename, frame in [('fold_summary', pd.DataFrame(overview)),
                            ('language_counts', pd.DataFrame(language_rows)),
                            ('support_by_fold_label', per_label), ('support_ranges', across),
                            ('rare_EN_candidates', priority), ('unseen_multilingual', unseen)]:
        frame.to_csv(dest / (filename + '.csv'), index=False, encoding='utf-8-sig')
    manifest.to_csv(dest / 'split_manifest_v2.csv', index=False)
    setup.save_json(dest / 'taxonomy.json', taxonomy)
    summary = {'status': 'MULTILINGUAL_SUPPORT_OK', 'folds': 3,
        'fine_classes': len(classes), 'specific_fine_classes': int(specific.sum()),
        'fold_summary': overview, 'validation_and_outer_unchanged': True,
        'heldout_labels_used_for_support_analysis': False, 'new_training_performed': False,
        'model_selected': False, 'output_directory': str(dest),
        'limitations': ['Counts measure documents, not independent information or transfer quality.',
            'Cross-lingual translations and undetected paraphrases are not ruled out by the lexical audit.',
            'Support bands are descriptive; 10+ does not imply sufficient training data.',
            'Minimum and maximum summarize three overlapping training sets; counts must not be summed across folds.',
            'More examples do not guarantee better English evaluation performance.']}
    setup.save_json(dest / 'summary.json', summary)
    print('Fine-label support audit: global Other excluded from specific-label counts.')
    display(pd.DataFrame(overview))
    print('Training documents by language and fold:')
    display(pd.DataFrame(language_rows).pivot(index='fold', columns='language', values='training_documents'))
    print('Top 15 classes with at most four English examples in any fold, ranked by minimum support gain:')
    with pd.option_context('display.max_colwidth', None):
        display(priority.head(15))
        print('Labels absent from multilingual training, by fold:')
        display(unseen[['fold', 'label']])
    archive_path = dest.with_suffix('.zip')
    with ZipFile(archive_path, 'w', ZIP_DEFLATED) as archive:
        for path in sorted(dest.iterdir()):
            archive.write(path, path.name)
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print('Saved artifact archive:', archive_path)
    return summary


multilingual_support_result = audit_multilingual_support()


Supporto delle categorie fine specifiche: Other globale esclusa dai conteggi delle classi.


,fold,EN_train,multilingual_train,auxiliary_documents,specific_labels_seen_EN,specific_labels_seen_multilingual,unseen_EN_become_seen,still_unseen_multilingual,labels_with_1_to_4_EN_examples,of_these_reach_5plus,of_these_reach_10plus
0,1,213,1502,1289,80,94,14,1,46,34,25
1,2,212,1501,1289,75,94,19,1,41,34,25
2,3,214,1503,1289,77,93,16,2,42,34,24


Documenti disponibili per lingua e fold:


language,BG,EN,HI,PT,RU
fold,,,,,
1,400,213,359,397,133
2,400,212,359,397,133
3,400,214,359,397,133


Prime 15 categorie con <=4 esempi EN in almeno un fold, ordinate per incremento minimo:


,label,EN_support_min,EN_support_max,multilingual_support_min,multilingual_support_max,added_support_min
0,CC: Amplifying Climate Fears: Amplifying existing fears of global warming,2,3,175,176,173
83,URW: Praise of Russia: Praise of Russian military might,4,6,137,139,133
3,CC: Amplifying Climate Fears: Other,0,1,103,104,103
85,URW: Praise of Russia: Russia is a guarantor of peace and prosperity,1,2,98,99,97
55,URW: Discrediting Ukraine: Discrediting Ukrainian military,3,4,97,98,94
61,URW: Discrediting Ukraine: Ukraine is a puppet of the West,4,9,96,101,92
84,URW: Praise of Russia: Russia has international support from a number of countries and people,1,2,75,76,74
57,URW: Discrediting Ukraine: Other,0,0,70,70,70
87,URW: Russia is the Victim: Other,1,2,71,72,70
53,URW: Blaming the war on others rather than the invader: Ukraine is the aggressor,4,5,73,74,69


Categorie ancora assenti nel training multilingue, per fold:


,fold,label
95,1,CC: Controversy about green technologies: Nuclear energy is not climate-friendly
191,2,CC: Controversy about green technologies: Nuclear energy is not climate-friendly
229,3,CC: Questioning the measurements and science: Greenhouse effect/carbon dioxide do not drive climate change
287,3,CC: Controversy about green technologies: Nuclear energy is not climate-friendly


{
  "status": "MULTILINGUAL_SUPPORT_OK",
  "folds": 3,
  "fine_classes": 96,
  "specific_fine_classes": 95,
  "fold_summary": [
    {
      "fold": 1,
      "EN_train": 213,
      "multilingual_train": 1502,
      "auxiliary_documents": 1289,
      "specific_labels_seen_EN": 80,
      "specific_labels_seen_multilingual": 94,
      "unseen_EN_become_seen": 14,
      "still_unseen_multilingual": 1,
      "labels_with_1_to_4_EN_examples": 46,
      "of_these_reach_5plus": 34,
      "of_these_reach_10plus": 25
    },
    {
      "fold": 2,
      "EN_train": 212,
      "multilingual_train": 1501,
      "auxiliary_documents": 1289,
      "specific_labels_seen_EN": 75,
      "specific_labels_seen_multilingual": 94,
      "unseen_EN_become_seen": 19,
      "still_unseen_multilingual": 1,
      "labels_with_1_to_4_EN_examples": 41,
      "of_these_reach_5plus": 34,
      "of_these_reach_10plus": 25
    },
    {
      "fold": 3,
      "EN_train": 214,
      "multilingual_train": 1503,
      "aux

## Persist the multilingual follow-up configuration

Records a separate planned multilingual embedding experiment as an immutable, content-addressed JSON configuration. This section defines the experiment only; it performs no embedding extraction or classifier training.


In [9]:
from pathlib import Path
import json
import hashlib

MULTILINGUAL_CONFIG = {
    'experiment': 'multilingual_encoder_SVM_EN_vs_MULTI_v1',
    'development_status': 'exploratory; follows prior outer analyses',
    'split_version': 'v2', 'taxonomy_version': 'project_v1_96fine',
    'training_seed': 42, 'folds': [1, 2, 3],
    'model_id': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'revision': '4328cf26390c98c5e3c738b4460a05b95f4911f5',
    'backbone': 'XLMRobertaModel', 'embedding_dimension': 768,
    'encoder_frozen': True, 'encoder_mode': 'eval',
    'encoder_precision': 'float32', 'attention': 'eager',
    'extraction_batch_size': 2,
    'max_length': 128, 'length_includes_special_tokens': True,
    'chunk_overlap_tokens': 0, 'chunk_policy': 'all content tokens, contiguous tokenizer chunks, no document truncation',
    'pooling': 'attention-mask mean pooling including special tokens',
    'chunk_normalization': 'L2',
    'document_aggregation': 'content-token-weighted mean of normalized chunks, then L2',
    'aggregation_note': 'project-defined document representation, not the default single-sequence model pipeline',
    'feature_standardization': None,
    'training_conditions': {
        'EN_only': ['EN'],
        'EN_plus_auxiliary': ['EN', 'BG', 'HI', 'PT', 'RU']
    },
    'validation_language': 'EN', 'outer_language': 'EN',
    'auxiliary_filter': 'existing make_fold grouping and exclusions; no new split',
    'classifier': {'C': 1.0, 'penalty': 'l2', 'loss': 'squared_hinge',
        'dual': 'auto', 'tol': 0.0001, 'max_iter': 10000,
        'class_weight': None, 'random_state': 42},
    'threshold_grid': [-1.0, -.9, -.8, -.7, -.6, -.5, -.4, -.3, -.2, -.1, 0., .1, .2, .3, .5],
    'selection': 'sample-F1 on inner validation, separately per level, fold and training condition',
    'threshold_tie_break': 'larger threshold', 'fixed_threshold_comparator': 0.0,
    'decoder': 'strictly greater than threshold; argmax fallback separately per level',
    'constant_targets': 'all-negative -> negative infinity; all-positive -> positive infinity',
    'hierarchy_postprocessing': False, 'Other_exclusive': False,
    'refit_train_plus_validation': False,
    'primary_metric': 'sample_F1_fine_project_96',
    'planned_systems': 2, 'planned_classifier_fits': 6,
    'labels_used_during_embedding_extraction': False,
    'course_test_used_for_modeling': False,
    'limitations': [
        'More auxiliary documents change sample size and label/domain distributions together.',
        'Fixed C preserves the solver recipe, not identical effective regularization as sample size changes.',
        'Lexical duplicate audit does not rule out translations or all paraphrases.',
        'Comparison with prior English MPNet also changes encoder and chunk length; it is not a pure data ablation.',
        'One training seed and exploratory outer evaluation.'
    ]
}

output = Path('/kaggle/working/nlp_setup/multilingual')
output.mkdir(parents=True, exist_ok=True)
serialized = json.dumps(MULTILINGUAL_CONFIG, ensure_ascii=False, sort_keys=True, indent=2)
digest = hashlib.sha256(serialized.encode('utf-8')).hexdigest()
config_path = output / ('experiment_config_' + digest[:12] + '.json')
if config_path.exists():
    assert json.loads(config_path.read_text(encoding='utf-8')) == MULTILINGUAL_CONFIG
else:
    config_path.write_text(serialized, encoding='utf-8')
print(json.dumps(MULTILINGUAL_CONFIG, ensure_ascii=False, indent=2))
print('MULTILINGUAL_CONFIG_OK — two conditions, three folds; no downloads, embedding extraction, or training.')
print('Saved experiment configuration:', config_path)
print('Configuration SHA-256:', digest)


{
  "experiment": "multilingual_encoder_SVM_EN_vs_MULTI_v1",
  "development_status": "exploratory; follows prior outer analyses",
  "split_version": "v2",
  "taxonomy_version": "project_v1_96fine",
  "training_seed": 42,
  "folds": [
    1,
    2,
    3
  ],
  "model_id": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
  "revision": "4328cf26390c98c5e3c738b4460a05b95f4911f5",
  "backbone": "XLMRobertaModel",
  "embedding_dimension": 768,
  "encoder_frozen": true,
  "encoder_mode": "eval",
  "encoder_precision": "float32",
  "attention": "eager",
  "extraction_batch_size": 2,
  "max_length": 128,
  "length_includes_special_tokens": true,
  "chunk_overlap_tokens": 0,
  "chunk_policy": "all content tokens, contiguous tokenizer chunks, no document truncation",
  "pooling": "attention-mask mean pooling including special tokens",
  "chunk_normalization": "L2",
  "document_aggregation": "content-token-weighted mean of normalized chunks, then L2",
  "aggregation_note": "project-